# Cars Dataset Cleaning and Preprocessing

### Import

In [11]:
import pandas as pd

cars = pd.read_parquet("data/processed/cars_merged.parquet")

print(cars.shape)
print(cars.columns)
print(cars.info())
cars.head()

(53655, 22)
Index(['source', 'listing_id', 'title', 'make', 'model', 'year', 'kms',
       'price', 'body_type', 'transmission', 'fuel_type', 'engine_l',
       'drive_type', 'location', 'state', 'used_or_new', 'status', 'variant',
       'series', 'cc', 'color', 'seats'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53655 entries, 0 to 53654
Data columns (total 22 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   source        53655 non-null  object 
 1   listing_id    17043 non-null  float64
 2   title         33068 non-null  object 
 3   make          53655 non-null  object 
 4   model         53655 non-null  object 
 5   year          53655 non-null  int32  
 6   kms           53655 non-null  int64  
 7   price         53655 non-null  float64
 8   body_type     53655 non-null  object 
 9   transmission  53655 non-null  object 
 10  fuel_type     53651 non-null  object 
 11  engine_l      51213 non-null  f

,source,listing_id,title,make,model,year,kms,price,body_type,transmission,...,drive_type,location,state,used_or_new,status,variant,series,cc,color,seats
0,raw_1,NaN,None,alfa,romeo 4c,2016,27011,97644.0,Convertible,Sports Automatic Dual Clutch,...,None,Queensland,None,Used,None,None,None,NaN,None,NaN
1,raw_1,NaN,None,alfa,romeo stelvio,2023,3909,85400.0,SUV,Sports Automatic,...,None,Victoria,None,Demo,None,None,None,NaN,None,NaN
2,raw_1,NaN,None,alfa,romeo tonale,2023,22,65250.0,SUV,Sports Automatic Dual Clutch,...,None,Victoria,None,Demo,None,None,None,NaN,None,NaN
3,raw_1,NaN,None,alfa,romeo tonale,2024,20,75990.0,SUV,Sports Automatic Dual Clutch,...,None,Victoria,None,New,None,None,None,NaN,None,NaN
4,raw_1,NaN,None,aston,martin rapide,2014,66993,145890.0,Hatchback,Sports Automatic,...,None,New South Wales,None,Used,None,None,None,NaN,None,NaN


### Missing Values

In [13]:
cars.isna().sum().sort_values(ascending=False)

state           37630
drive_type      37630
seats           36612
listing_id      36612
color           36612
series          36612
variant         36612
status          36612
cc              22206
title           20587
location        17043
engine_l         2442
fuel_type           4
transmission        0
body_type           0
used_or_new         0
price               0
kms                 0
year                0
model               0
make                0
source              0
dtype: int64

In [14]:
(cars.isna().sum() / len(df) * 100).sort_values(ascending=False)

state           70.133259
drive_type      70.133259
seats           68.235952
listing_id      68.235952
color           68.235952
series          68.235952
variant         68.235952
status          68.235952
cc              41.386637
title           38.369211
location        31.764048
engine_l         4.551300
fuel_type        0.007455
transmission     0.000000
body_type        0.000000
used_or_new      0.000000
price            0.000000
kms              0.000000
year             0.000000
model            0.000000
make             0.000000
source           0.000000
dtype: float64

### Describing values

In [16]:
cars.describe()

,listing_id,year,kms,price,engine_l,cc,seats
count,1.704300e+04,53655.000000,53655.000000,53655.000000,51213.000000,31449.000000,17043.000000
mean,1.279026e+07,2017.819849,80606.710465,40025.898369,2.392029,1352.197272,5.116353
std,5.051347e+04,5.146029,74902.710210,28652.094984,0.871974,1398.876856,1.121382
min,1.153013e+07,1980.000000,1.000000,900.000000,0.700000,0.000000,2.000000
25%,1.275715e+07,2015.000000,16624.500000,22163.000000,1.991000,4.000000,5.000000
50%,1.280207e+07,2019.000000,65022.000000,33980.000000,2.179000,1498.000000,5.000000
75%,1.283132e+07,2022.000000,121722.500000,48990.000000,2.800000,2440.000000,5.000000
max,1.285246e+07,2025.000000,987475.000000,499990.000000,7.300000,7300.000000,14.000000


### Column Inspections

In [17]:
for col in cars.select_dtypes(include='object'):
    print(col, cars[col].nunique())

source 3
title 16942
make 78
model 1039
body_type 62
transmission 13
fuel_type 15
drive_type 5
location 626
state 8
used_or_new 9
status 3
variant 2361
series 2286
color 235


In [18]:
cars['fuel_type'].value_counts()

Diesel                       17322
Petrol                       11836
Unleaded Petrol               7015
Unleaded                      6847
Premium Unleaded Petrol       3434
Premium                       3230
Hybrid                        2077
Electric                       831
-                              534
Unleaded Petrol/Electric       255
Premium Unleaded/Electric      194
Liquid Petroleum Gas            40
Diesel/Electric                 18
LPG                             15
Leaded                           3
Name: fuel_type, dtype: int64

### Handling missing values

#### Location and States

In [19]:
cars['location'].dropna().sample(10)

16892             Berwick, VIC
39027         Morningside, QLD
24107               Queensland
8126           New South Wales
15238          New South Wales
52351          New South Wales
8867           Stuart Park, NT
28504           Southport, QLD
29726            Lidcombe, NSW
25567    Albion Park Rail, NSW
Name: location, dtype: object

In [22]:
# Extract state from string
cars['state_from_location'] = cars['location'].str.split(',').str[-1].str.strip()

# Combine both columns
cars['state'] = cars['state'].fillna(cars['state_from_location'])

# Drop helper column
cars = cars.drop(columns=['state_from_location'])

In [32]:
# Kaggle's "Australia Car Market" dataset 
#(https://www.kaggle.com/datasets/lainguyn123/australia-car-market-data/data)
# does not include state or location, to preserve this information, just set as unknown
cars['state'] = cars['state'].fillna('Unknown')
cars['location'] = cars['location'].fillna('Unknown')

#### Low Value Columns - dropping

In [ ]:
# Each data row has model + make, so series, title and listing_id don't help
cars = cars.drop(columns=['title', 'series', 'listing_id'])

#### Fill categorical and numerical missing values
To fill NaN in fuel type, seats, cc, and engine l, we need to standardize the names of all the vehicles to make sure same cars are lsited as the same name. Then we group-based impute them.
For 'drive_type', 'color', 'variant', 'status' we can just fill with "Unknown".

In [26]:
cat_cols = ['drive_type', 'color', 'variant', 'status']

for col in cat_cols:
    cars[col] = cars[col].fillna('Unknown')

In [40]:
cars['make'].unique()

array(['alfa', 'aston', 'audi', 'bmw', 'chery', 'chevrolet', 'chrysler',
       'citroen', 'cupra', 'daihatsu', 'dodge', 'fiat', 'ford', 'fpv',
       'genesis', 'great', 'gwm', 'haval', 'holden', 'honda', 'hsv',
       'hyundai', 'isuzu', 'jac', 'jaguar', 'jeep', 'kgm', 'kia', 'land',
       'land rover', 'ldv', 'lexus', 'mahindra', 'mazda', 'mercedes-benz',
       'mg', 'mini', 'mitsubishi', 'nissan', 'opel', 'peugeot', 'porsche',
       'ram', 'renault', 'saab', 'skoda', 'smart', 'ssangyong', 'subaru',
       'suzuki', 'tesla', 'toyota', 'volkswagen', 'volvo', 'great wall',
       'aston martin', 'abarth', 'alfa romeo', 'lamborghini', 'foton',
       'infiniti', 'bentley', 'maserati', 'mclaren', 'byd', 'iveco',
       'rover', 'geely', 'hummer', 'ferrari', 'ineos', 'tata', 'daewoo',
       'hino', 'proton', 'rolls-royce', 'polestar', 'mitsubishi fuso'],
      dtype=object)

#### Fixing make names

In [41]:
make_map = {
    'aston': 'aston martin',
    'great': 'great wall',
    'gwm': 'great wall',
    'land': 'land rover'
}

cars['make'] = cars['make'].replace(make_map)

#### Fixing model names